In [1]:
import sys
from pathlib import Path

# Get project root - the project root should have both 'vault' and 'ensemble' directories
# Walk up from current directory until we find a directory with both
current_dir = Path.cwd()
project_root = current_dir

# Walk up the directory tree to find project root
while project_root != project_root.parent:
    # Check if this directory has both vault and ensemble (indicating project root)
    has_vault = (project_root / 'vault').exists()
    has_ensemble = (project_root / 'ensemble').exists()
    
    if has_vault and has_ensemble:
        # Found the project root
        break
    
    project_root = project_root.parent
else:
    # If we reached the filesystem root without finding it, use parent of current_dir
    # (assuming we're in research/ subdirectory)
    if (current_dir.parent / 'vault').exists() and (current_dir.parent / 'ensemble').exists():
        project_root = current_dir.parent
    else:
        # Last resort: use current directory
        project_root = current_dir

# Add project root to Python path
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f"Current directory: {current_dir}")
print(f"Project root: {project_root}")
print(f"Has vault: {(project_root / 'vault').exists()}")
print(f"Has ensemble: {(project_root / 'ensemble').exists()}")
print(f"Python path includes: {str(project_root) in sys.path}")

Current directory: /home/raman/repos/Trading-Algo/research
Project root: /home/raman/repos/Trading-Algo
Has vault: True
Has ensemble: True
Python path includes: True


In [2]:
"""
Portfolio Testing with PortfolioTester

This notebook demonstrates how to use the PortfolioTester framework to:
1. Load data and create portfolios
2. Fit portfolios using candles
3. Generate predictions at different granularity levels
4. Calculate returns and generate tearsheets
5. Analyze performance at portfolio, ensemble, and base model levels
"""

from datetime import datetime
import pandas as pd
import numpy as np

from utils.enums import TimeFrame, Ticker
from utils import helpers
from ensemble.portfolio import Portfolio
from ensemble.diversified_ensemble import DiversifiedEnsemble
from ensemble.portfolio_tester import PortfolioTester
from ensemble.weight_layer import WeightLayer

# Portfolio Testing Framework Demo

## Overview

The `PortfolioTester` class provides a comprehensive framework for testing portfolios:
- **Automatic returns calculation**: Calculates log returns from candles automatically
- **Granular predictions**: Get predictions at portfolio, ensemble, and base model levels
- **Integrated tearsheets**: Generate QuantStats tearsheets at all granularity levels
- **Baseline comparison**: Compare against equal-weight or buy-and-hold baselines

In [3]:
# Load data
print("Loading data...")



tickers = [ Ticker.ES, Ticker.NQ, Ticker.YM, Ticker.RTY, Ticker.TLT]




train_candles = helpers.load_data_multi_ticker(
    tickers=tickers,
    timeframe=TimeFrame.D,
    start=datetime(2005, 1, 1),
    end=datetime(2020, 1, 1),
    use_millisecond_offset=True
)

test_candles = helpers.load_data_multi_ticker(
    tickers=tickers,
    timeframe=TimeFrame.D,
    start=datetime(2020, 1, 1),
    end=datetime(2024, 1, 1),
    use_millisecond_offset=True
)

print(f"Training data: {len(train_candles)} candles")
print(f"Test data: {len(test_candles)} candles")
print(f"Tickers: {sorted(train_candles['ticker'].unique())}")

Loading data...
Training data: 12992 candles
Test data: 5038 candles
Tickers: [<Ticker.ES: 'US500'>, <Ticker.NQ: 'US100'>, <Ticker.YM: 'US30'>, <Ticker.RTY: 'RUSSELL_2000'>, <Ticker.TLT: 'TWENTY_YR_NOTE'>]


## Step 1: Create Portfolio

For this demo, we'll use a simple buy-and-hold ensemble. In practice, you would load your ensemble from the vault or create it from feature configs.

In [4]:
# Load ensemble from vault using vault_manager
from ensemble.vault_manager import load_ensemble_from_vault

# Load ensembles from vault
USE_CACHE = True
TARGET_VOL = 0.15

ensemble_dirs = {
    'buy_hold': 'vault/D/buy-hold_indices_long',
    'mean_reversion_long': 'vault/D/mean-reversion_indices_long',
    'seasonal_indices_eom': 'vault/D/seasonal_indices_eom_long',
    'seasonal_bonds_month': 'vault/D/seasonal_bonds_month_long_short',
}


def _enable_cache(loaded_ensemble: DiversifiedEnsemble) -> DiversifiedEnsemble:
    loaded_ensemble.use_cache = USE_CACHE
    _ = [setattr(model, 'use_cache', USE_CACHE) for model in loaded_ensemble.base_models.values()]
    return loaded_ensemble


ensembles = {
    name: _enable_cache(
        load_ensemble_from_vault(path, refit=True, target_volatility=TARGET_VOL)
    )
    for name, path in ensemble_dirs.items()
}

print("Loaded ensembles:")
_ = [print(f"  {name}: {ensemble_dir}") for name, ensemble_dir in ensemble_dirs.items()]

# Weight layer: combine forecasts using inverse_correlation or sortino_optimized
WEIGHT_METHOD = 'sortino_optimized'  # or 'inverse_correlation'
weight_layer = WeightLayer(
    weight_method=WEIGHT_METHOD,
    fdm_max=2.5,
    # Sortino-optimized params (only used when weight_method='sortino_optimized')
    gamma=0.5,                # risk aversion (lower = let optimizer differentiate more)
    shrinkage_lambda=0.02,      # penalty toward equal weights (lower = allow more concentration)
    mean_shrinkage=1,       # shrink expected returns (higher = trust activity-adjusted means more)
    min_history=30,            # min observations before optimizing
    activity_adjust_means=True,  # use mean forecast when active (avoids always-on e.g. buy_hold dominating)
)

# Create portfolio
portfolio = Portfolio(
    ensembles=list(ensembles.values()),
    trading_timeframe=TimeFrame.D,
    target_volatility=TARGET_VOL,
    max_position_pct=3.5,
    weight_layer=weight_layer,
    use_cache=USE_CACHE
)
print(f"\nPortfolio created with {len(portfolio.ensembles)} ensemble(s), weight_method={WEIGHT_METHOD}")

Loaded ensembles:
  buy_hold: vault/D/buy-hold_indices_long
  mean_reversion_long: vault/D/mean-reversion_indices_long
  seasonal_indices_eom: vault/D/seasonal_indices_eom_long
  seasonal_bonds_month: vault/D/seasonal_bonds_month_long_short

Portfolio created with 4 ensemble(s), weight_method=sortino_optimized


## Step 2: Initialize PortfolioTester

The PortfolioTester wraps your portfolio and provides a convenient API for testing.

In [5]:
# Initialize PortfolioTester
tester = PortfolioTester(
    portfolio=portfolio,
    baseline_mode='equal_weight'  # or 'buy_hold'
)

# Fit portfolio (automatically calculates log returns)
print("\nFitting portfolio...")
tester.fit(train_candles)

# Print comprehensive diagnostics
tester.portfolio.print_diagnostics()

# Also print individual values for easy access
diagnostics = tester.portfolio.get_diagnostics()
wl = diagnostics.get('weight_layer') or {}
summary = wl.get('summary') or {}
print(f"\n📋 Quick Summary:")
print(f"  IDM: {diagnostics.get('idm', 'Not calculated')}")
print(f"  FDM (mean): {diagnostics.get('fdm', summary.get('mean_fdm', 'Not calculated'))}")
print(f"  Weight method: {wl.get('weight_method', 'N/A')}")
print(f"  Mean Return Correlation: {diagnostics.get('mean_return_correlation', 'N/A')}")
print(f"  Mean Forecast Correlation: {diagnostics.get('mean_forecast_correlation', 'N/A')}")
if summary:
    print(f"  Tickers (weight layer): {summary.get('n_tickers', 0)}, models/ticker: {summary.get('mean_models_per_ticker', 0):.1f}")


Partial cache coverage for buy_hold (RTY, D). Cache starts at 2017-06-02 00:00:00, requested 2005-01-03 00:00:00. Proceeding with available data.
Partial cache coverage for cumulative_rsi (RTY, D). Cache starts at 2017-06-02 00:00:00, requested 2005-01-03 00:00:00. Proceeding with available data.
Partial cache coverage for ultimate_c (RTY, D). Cache starts at 2017-06-02 00:00:00, requested 2005-01-03 00:00:00. Proceeding with available data.
Partial cache coverage for rsi_regime (RTY, D). Cache starts at 2017-06-02 00:00:00, requested 2005-01-03 00:00:00. Proceeding with available data.



Fitting portfolio...


Partial cache coverage for rsi (RTY, D). Cache starts at 2017-06-02 00:00:00, requested 2005-01-03 00:00:00. Proceeding with available data.
Partial cache coverage for rsi (RTY, D). Cache starts at 2017-06-02 00:00:00, requested 2005-01-03 00:00:00. Proceeding with available data.
BaseModel.vectorized_predict() has 844 NaN values. Filling with 0.0.
BaseModel.vectorized_predict() has 778 NaN values. Filling with 0.0.
BaseModel.vectorized_predict() has 849 NaN values. Filling with 0.0.
BaseModel.vectorized_predict() has 384 NaN values. Filling with 0.0.
Partial cache coverage for seasonalindiceseof (RTY, D). Cache starts at 2017-06-02 00:00:00, requested 2005-01-03 00:00:00. Proceeding with available data.
BaseModel.vectorized_predict() has 844 NaN values. Filling with 0.0.
BaseModel.vectorized_predict() has 778 NaN values. Filling with 0.0.



Fitting WeightLayer...


BaseModel.vectorized_predict() has 849 NaN values. Filling with 0.0.
BaseModel.vectorized_predict() has 384 NaN values. Filling with 0.0.
Found constant forecast columns: ['buy_hold_signal_D::quantile_binning_2']. Treated as perfectly correlated (corr=1.0) for FDM.



✓ WeightLayer fitted successfully!
  Tickers: 5
  Mean FDM: 1.7794 (range: 1.0000 - 2.2968)
  Mean models per ticker: 5.8
  Per-ticker FDM:
    Ticker.ES: FDM=1.4044 (7 model(s))
    Ticker.NQ: FDM=2.0700 (7 model(s))
    Ticker.YM: FDM=2.1257 (7 model(s))
    Ticker.RTY: FDM=2.2968 (7 model(s))
    Ticker.TLT: FDM=1.0000 (1 model(s))

PORTFOLIO DIAGNOSTICS

📊 Portfolio Status:
  Fitted: True
  Trading Timeframe: D
  Number of Instruments: 5
  Instruments: ['ES', 'NQ', 'YM', 'TLT', 'RTY']

🎯 Instrument Diversification Multiplier (IDM):
  IDM: 1.3675032413859882
  IDM Max: 2.5
  Mean Return Correlation: 0.524740738101207

🔮 Forecast Diversification Multiplier (FDM):
  Mean FDM: 1.779378599858608
  FDM Range: 1.0 - 2.2968415946598983
  FDM Max: 2.5
  Number of Tickers: 5
  Mean Models per Ticker: 5.8

⚖️  Weight Layer:
  Fitted: True
  Weight Method: sortino_optimized

  Per-Ticker FDM and Models:
    Ticker.ES:
      FDM: 1.4044
      Models: 7 (buy_hold_signal_D::quantile_binning_2, c

## Step 3: Generate Predictions

You can get predictions at different granularity levels:
- **Portfolio level**: Aggregated positions (default)
- **Ensemble level**: Individual ensemble predictions
- **Base model level**: Individual base model predictions

In [6]:
# Basic prediction (portfolio level only)
print("\nGenerating portfolio-level predictions...")
positions = tester.predict(test_candles)
print(f"Generated {len(positions)} position predictions")
print(f"Position fraction range: [{positions['position_fraction'].min():.4f}, {positions['position_fraction'].max():.4f}]")


BaseModel.vectorized_predict() has 295 NaN values. Filling with 0.0.
BaseModel.vectorized_predict() has 276 NaN values. Filling with 0.0.
BaseModel.vectorized_predict() has 310 NaN values. Filling with 0.0.
BaseModel.vectorized_predict() has 492 NaN values. Filling with 0.0.



Generating portfolio-level predictions...
Generated 5038 position predictions
Position fraction range: [-0.3881, 0.5470]


In [7]:

# With ensemble-level and base model-level granularity
print("\nGenerating granular predictions (ensemble + base model level)...")
results = tester.predict(
    test_candles,
    return_ensemble_predictions=True,
    return_base_model_predictions=True
)

portfolio_positions = results['portfolio']
ensemble_predictions = results['ensembles']
base_model_predictions = results['base_models']

print(f"\nGranular predictions:")
print(f"  Portfolio positions: {len(portfolio_positions)}")
print(f"  Ensemble predictions: {len(ensemble_predictions)}")
print(f"  Base model predictions: {len(base_model_predictions)}")

BaseModel.vectorized_predict() has 295 NaN values. Filling with 0.0.
BaseModel.vectorized_predict() has 276 NaN values. Filling with 0.0.
BaseModel.vectorized_predict() has 310 NaN values. Filling with 0.0.
BaseModel.vectorized_predict() has 492 NaN values. Filling with 0.0.



Generating granular predictions (ensemble + base model level)...

Granular predictions:
  Portfolio positions: 5038
  Ensemble predictions: 4
  Base model predictions: 8


In [8]:
portfolio_positions

,ticker,datetime,forecast_score,position_fraction
0,Ticker.ES,2020-01-02,0.431370,0.117980
1,Ticker.NQ,2020-01-02,0.000000,0.000000
2,Ticker.YM,2020-01-02,0.000000,0.000000
3,Ticker.RTY,2020-01-02,0.000000,0.000000
4,Ticker.TLT,2020-01-02,0.000000,0.000000
...,...,...,...,...
5033,Ticker.ES,2023-12-29,0.890308,0.243500
5034,Ticker.NQ,2023-12-29,2.000000,0.547001
5035,Ticker.YM,2023-12-29,1.393263,0.381058
5036,Ticker.RTY,2023-12-29,1.897147,0.518871


## Step 4: Calculate Returns

Calculate strategy returns and baseline returns for comparison.

In [9]:
# Calculate strategy returns
print("\nCalculating strategy returns...")
strategy_returns = tester.calculate_strategy_returns(test_candles)
print(f"Strategy returns: {len(strategy_returns)} days")
print(f"Total return: {strategy_returns.sum():.4f}")
if strategy_returns.std() > 0:
    sharpe = strategy_returns.mean() / strategy_returns.std() * np.sqrt(252)
    print(f"Annualized Sharpe: {sharpe:.4f}")

# Calculate baseline returns
print("\nCalculating baseline returns...")
baseline_returns = tester.calculate_baseline_returns(test_candles)
print(f"Baseline returns: {len(baseline_returns)} days")
print(f"Total return: {baseline_returns.sum():.4f}")
if baseline_returns.std() > 0:
    baseline_sharpe = baseline_returns.mean() / baseline_returns.std() * np.sqrt(252)
    print(f"Annualized Sharpe: {baseline_sharpe:.4f}")

# Compare performance
print(f"\nPerformance Comparison:")
print(f"  Strategy total return: {strategy_returns.sum():.4f} ({strategy_returns.sum()*100:.2f}%)")
print(f"  Baseline total return: {baseline_returns.sum():.4f} ({baseline_returns.sum()*100:.2f}%)")
print(f"  Excess return: {(strategy_returns.sum() - baseline_returns.sum()):.4f} ({(strategy_returns.sum() - baseline_returns.sum())*100:.2f}%)")


Calculating strategy returns...
Strategy returns: 1007 days
Total return: 1.1016
Annualized Sharpe: 1.1582

Calculating baseline returns...
Baseline returns: 5033 days
Total return: 1.1943
Annualized Sharpe: 0.2457

Performance Comparison:
  Strategy total return: 1.1016 (110.16%)
  Baseline total return: 1.1943 (119.43%)
  Excess return: -0.0927 (-9.27%)


## Step 5: Generate Tearsheets

Generate QuantStats tearsheets for performance analysis.

In [10]:
# Generate portfolio-level tearsheet
print("\nGenerating portfolio tearsheet...")
tester.generate_tearsheet(
    strategy_name='Buy Hold Portfolio',
    output_dir='reports/portfolio',  # Save HTML report to this directory
    mode='html',  # 'html', 'full', 'basic', or 'metrics'
    candles_df=test_candles
)

# Generate ensemble-level tearsheets (if available)
if tester.ensemble_predictions:
    print("\nGenerating ensemble tearsheets...")
    tester.generate_ensemble_tearsheets(
        output_dir='reports/ensembles',  # Save HTML reports to this directory
        mode='html',
        candles_df=test_candles
    )

# Generate base model-level tearsheets (if available)
if tester.base_model_predictions:
    print("\nGenerating base model tearsheets...")
    tester.generate_base_model_tearsheets(
        output_dir='reports/base_models',  # Save HTML reports to this directory
        mode='html',
        candles_df=test_candles
    )


Generating portfolio tearsheet...


findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font f


[OK] HTML tearsheet saved to: reports/portfolio/Buy_Hold_Portfolio_tearsheet.html
   Note: Using non-compounded returns (Robert Carver methodology)

Generating ensemble tearsheets...


findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font f


[OK] HTML tearsheet saved to: reports/ensembles/ensemble_0_tearsheet.html
   Note: Using non-compounded returns (Robert Carver methodology)


...


[OK] HTML tearsheet saved to: reports/base_models/ensemble_1_rsi_signal_D_lookback_[5]_quantile_binning_10_tearsheet.html
   Note: Using non-compounded returns (Robert Carver methodology)


findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font f


[OK] HTML tearsheet saved to: reports/base_models/ensemble_2_seasonal_indices_eof_signal_D_rule_based_binning_1_tearsheet.html
   Note: Using non-compounded returns (Robert Carver methodology)


findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font f


[OK] HTML tearsheet saved to: reports/base_models/ensemble_3_seasonal_bonds_month_signal_D_longDays_7_mode_short_flat_long_shortDays_7_rule_based_binning_1_tearsheet.html
   Note: Using non-compounded returns (Robert Carver methodology)


In [11]:
# Generate training-set tearsheets (predict on train period, then restore test state)
print("\nGenerating training-set tearsheets...")

from ensemble.portfolio_tester import calculate_strategy_returns_from_positions, calculate_baseline_returns

# Get positions for training period (overwrites tester state temporarily)
tester.predict(
    train_candles,
    return_ensemble_predictions=bool(tester.ensemble_predictions),
    return_base_model_predictions=bool(tester.base_model_predictions),
)

train_strategy_returns = calculate_strategy_returns_from_positions(
    tester.positions_df, train_candles, strategy='long'
)
train_baseline_returns = calculate_baseline_returns(
    train_candles, equal_weight=(tester.baseline_mode == 'equal_weight')
)

tester.generate_tearsheet(
    strategy_name='Buy Hold Portfolio (Train)',
    output_dir='reports/portfolio_train',
    mode='html',
    strategy_returns=train_strategy_returns,
    baseline_returns=train_baseline_returns,
)

if tester.ensemble_predictions:
    print("\nGenerating training ensemble tearsheets...")
    tester.generate_ensemble_tearsheets(
        output_dir='reports/ensembles_train',
        mode='html',
        candles_df=train_candles,
        baseline_returns=train_baseline_returns,
    )

if tester.base_model_predictions:
    print("\nGenerating training base model tearsheets...")
    tester.generate_base_model_tearsheets(
        output_dir='reports/base_models_train',
        mode='html',
        candles_df=train_candles,
        baseline_returns=train_baseline_returns,
    )

# Restore test-set predictions for any later cells
tester.predict(
    test_candles,
    return_ensemble_predictions=bool(tester.ensemble_predictions),
    return_base_model_predictions=bool(tester.base_model_predictions),
)

BaseModel.vectorized_predict() has 844 NaN values. Filling with 0.0.
BaseModel.vectorized_predict() has 778 NaN values. Filling with 0.0.



Generating training-set tearsheets...


BaseModel.vectorized_predict() has 849 NaN values. Filling with 0.0.
BaseModel.vectorized_predict() has 384 NaN values. Filling with 0.0.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
f


[OK] HTML tearsheet saved to: reports/portfolio_train/Buy_Hold_Portfolio_(Train)_tearsheet.html
   Note: Using non-compounded returns (Robert Carver methodology)

Generating training ensemble tearsheets...


findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font f

...

findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font f


[OK] HTML tearsheet saved to: reports/base_models_train/ensemble_2_seasonal_indices_eof_signal_D_rule_based_binning_1_tearsheet.html
   Note: Using non-compounded returns (Robert Carver methodology)


findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font f


[OK] HTML tearsheet saved to: reports/base_models_train/ensemble_3_seasonal_bonds_month_signal_D_longDays_7_mode_short_flat_long_shortDays_7_rule_based_binning_1_tearsheet.html
   Note: Using non-compounded returns (Robert Carver methodology)


{'portfolio':           ticker   datetime  forecast_score  position_fraction
 0      Ticker.ES 2020-01-02        0.431370           0.117980
 1      Ticker.NQ 2020-01-02        0.000000           0.000000
 2      Ticker.YM 2020-01-02        0.000000           0.000000
 3     Ticker.RTY 2020-01-02        0.000000           0.000000
 4     Ticker.TLT 2020-01-02        0.000000           0.000000
 ...          ...        ...             ...                ...
 5033   Ticker.ES 2023-12-29        0.890308           0.243500
 5034   Ticker.NQ 2023-12-29        2.000000           0.547001
 5035   Ticker.YM 2023-12-29        1.393263           0.381058
 5036  Ticker.RTY 2023-12-29        1.897147           0.518871
 5037  Ticker.TLT 2023-12-29        1.418966           0.388088
 
 [5038 rows x 4 columns],
 'ensembles': {'ensemble_0':           ticker   datetime  forecast_score  position_fraction
  0      Ticker.ES 2020-01-02        1.561688           0.427123
  1      Ticker.NQ 2020-01-02     